# Precision AgriTech Platform - Multi-Dataset Retraining (Option A)
### Eliminating In-Field Domain Shift via Dual-Head MobileNetV3-Large

**Retraining Objectives:**
1. **Break Laboratory Background Bias:** Replace uniform gray paper backgrounds (PlantVillage) with authentic agricultural farm imagery (soil, weeds, shadows, outdoor sun) from Mendeley and PlantDoc.
2. **Dual-Head Multi-Task Architecture:** Predict Crop Species (4 classes) and Disease Pathology (8 classes) concurrently using shared MobileNetV3-Large representations.
3. **Zero Data Leakage:** SHA-256 deduplicated splitting (70% train / 15% val / 15% test) strictly balanced at 50% authentic field + 50% clean lab per class.

## 1. Hardware Verification & Reproducibility Setup

In [ ]:
!nvidia-smi
import random, os, json, time
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

## 2. Option A: Automated Multi-Dataset Onboarding & Ingestion

This cell automatically:
1. **Clones PlantVillage** (`spMohanty/PlantVillage-Dataset`) and **PlantDoc** (`pratikkayal/PlantDoc-Dataset`) from GitHub.
2. **Mounts Google Drive** to access your downloaded Mendeley datasets (Potato, Corn, Tomato).
3. **Auto-discovers and unpacks** your Mendeley zip archives into `/content/data/raw/`.
4. **Executes the Ingestion Engine**: verifies pixel integrity, hashes every image via SHA-256 for zero-leakage, enforces a strict 700-image per class cap, and balances 50% field + 50% lab.
5. Produces validated manifests in `/content/manifests/`.

In [ ]:
# Install required dependencies
%pip install -q torch torchvision torchaudio scikit-learn pillow matplotlib onnx onnxruntime

import os, sys, glob, zipfile, shutil, hashlib, random, logging
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Set, Union
from PIL import Image, UnidentifiedImageError

# 1. Clone Public Datasets from GitHub
if not Path("/content/PlantVillage-Dataset").exists():
    print("🌿 Cloning PlantVillage from GitHub...")
    !git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /content/PlantVillage-Dataset

if not Path("/content/PlantDoc-Dataset").exists():
    print("🌳 Cloning PlantDoc from GitHub...")
    !git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git /content/PlantDoc-Dataset

# 2. Mount Google Drive for Mendeley Datasets
from google.colab import drive
drive.mount('/content/drive')

# 3. Discover & Unpack Mendeley Zips from Google Drive
raw_dir = Path("/content/data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

def find_and_extract_zips(drive_base: Path):
    # Look for zip archives across Drive and /content
    all_zips = list(drive_base.rglob("*.zip")) + list(Path("/content").glob("*.zip"))
    print(f"🔍 Found {len(all_zips)} zip archive(s) in Drive / content.")
    
    targets = {
        "potato": ["potato", "mwbwbdzz7z"],
        "corn":   ["corn", "hmkd6nbngr", "maize"],
        "tomato": ["tomato", "zfv4jj7855"]
    }
    extracted_paths = {}
    
    for key, keywords in targets.items():
        dest = raw_dir / f"mendeley_{key}"
        if dest.exists() and any(dest.iterdir()):
            print(f"  ✓ {key.capitalize()} already unpacked at {dest}")
            extracted_paths[key] = dest
            continue
            
        matched_zip = next((z for z in all_zips if any(kw in z.name.lower() for kw in keywords)), None)
        if matched_zip:
            print(f"  📦 Unpacking {matched_zip.name} -> {dest}...")
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(matched_zip, 'r') as zf:
                zf.extractall(dest)
            print(f"  ✅ {key.capitalize()} unpacked successfully!")
            extracted_paths[key] = dest
        else:
            # Also check if already uploaded as an unzipped folder in Drive
            folder_match = next((d for d in drive_base.rglob("*") if d.is_dir() and any(kw in d.name.lower() for kw in keywords)), None)
            if folder_match:
                print(f"  📁 Found folder in Drive for {key}: {folder_match}")
                extracted_paths[key] = folder_match
            else:
                print(f"  ⚠️ Warning: No zip/folder found for {key} in Drive. Will backfill if missing.")
                extracted_paths[key] = None
    return extracted_paths

mendeley_paths = find_and_extract_zips(Path("/content/drive/MyDrive"))

# 4. Ingestion Engine Implementation
CANONICAL_CLASSES = [
    "Apple___Cedar_apple_rust", "Apple___healthy",
    "Corn_(maize)___Common_rust_", "Corn_(maize)___Northern_Leaf_Blight", "Corn_(maize)___healthy",
    "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
    "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___healthy",
]

CLASS_TO_MULTITASK = {
    "Apple___healthy":                     {"crop_idx": 0, "disease_idx": 0},
    "Apple___Cedar_apple_rust":            {"crop_idx": 0, "disease_idx": 6},
    "Corn_(maize)___healthy":              {"crop_idx": 1, "disease_idx": 0},
    "Corn_(maize)___Common_rust_":         {"crop_idx": 1, "disease_idx": 4},
    "Corn_(maize)___Northern_Leaf_Blight": {"crop_idx": 1, "disease_idx": 5},
    "Potato___healthy":                    {"crop_idx": 2, "disease_idx": 0},
    "Potato___Early_blight":               {"crop_idx": 2, "disease_idx": 1},
    "Potato___Late_blight":                {"crop_idx": 2, "disease_idx": 2},
    "Tomato___healthy":                    {"crop_idx": 3, "disease_idx": 0},
    "Tomato___Early_blight":               {"crop_idx": 3, "disease_idx": 1},
    "Tomato___Late_blight":                {"crop_idx": 3, "disease_idx": 2},
    "Tomato___Bacterial_spot":             {"crop_idx": 3, "disease_idx": 3},
}

MENDELEY_CORN_MAP = {
    "common rust": "Corn_(maize)___Common_rust_",
    "corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
    "healthy": "Corn_(maize)___healthy",
}

MENDELEY_POTATO_MAP = {
    "healthy": "Potato___healthy",
    "early leaf spot": "Potato___Early_blight",
    "early blight": "Potato___Early_blight",
    "late leaf spot": "Potato___Late_blight",
    "late blight": "Potato___Late_blight",
}

MENDELEY_TOMATO_MAP = {
    "tomato healthy": "Tomato___healthy",
    "healthy": "Tomato___healthy",
    "bacterial spot": "Tomato___Bacterial_spot",
    "early blight": "Tomato___Early_blight",
    "late blight": "Tomato___Late_blight",
}

PLANTDOC_MAP = {
    "apple leaf": "Apple___healthy",
    "apple rust leaf": "Apple___Cedar_apple_rust",
    "corn leaf": "Corn_(maize)___healthy",
    "corn rust leaf": "Corn_(maize)___Common_rust_",
    "corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
    "potato leaf": "Potato___healthy",
    "potato leaf early blight": "Potato___Early_blight",
    "potato leaf late blight": "Potato___Late_blight",
    "tomato leaf": "Tomato___healthy",
    "tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "tomato early blight leaf": "Tomato___Early_blight",
    "tomato leaf early blight": "Tomato___Early_blight",
    "tomato leaf late blight": "Tomato___Late_blight",
}

def calculate_sha256(fp: Path) -> str:
    h = hashlib.sha256()
    with open(fp, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

def discover_images(root: Path, mapping: Dict[str, str], domain: str, src_name: str, global_hashes: Set[str]):
    if not root or not root.exists():
        return []
    norm_map = {k.strip().lower().replace('_', ' '): v for k, v in mapping.items()}
    samples = []
    for d in [root] + [p for p in root.rglob('*') if p.is_dir()]:
        c_name = d.name.strip().lower().replace('_', ' ')
        canonical_cls = norm_map.get(c_name)
        if canonical_cls and canonical_cls in CANONICAL_CLASSES:
            enc = CLASS_TO_MULTITASK[canonical_cls]
            for f in d.glob('*.*'):
                if f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                    try:
                        with Image.open(f) as img:
                            img.load()
                            if img.size[0] < 100 or img.size[1] < 100:
                                continue
                        sha = calculate_sha256(f)
                        if sha not in global_hashes:
                            global_hashes.add(sha)
                            samples.append({
                                'file_path': str(f).replace('\\', '/'),
                                'canonical_class': canonical_cls,
                                'crop_idx': enc['crop_idx'],
                                'disease_idx': enc['disease_idx'],
                                'domain': domain,
                                'source_dataset': src_name,
                                'sha256': sha
                            })
                    except Exception:
                        continue
    return samples

print("\n🚀 Ingesting all multi-dataset sources...")
global_hashes = set()
collected = []

# PlantVillage (Lab)
pv_map = {c: c for c in CANONICAL_CLASSES}
pv_samples = discover_images(Path('/content/PlantVillage-Dataset/raw/color'), pv_map, 'lab', 'PlantVillage', global_hashes)
collected.extend(pv_samples)
print(f"  + Ingested {len(pv_samples)} lab samples from PlantVillage")

# PlantDoc (Field)
pd_samples = discover_images(Path('/content/PlantDoc-Dataset'), PLANTDOC_MAP, 'field', 'PlantDoc', global_hashes)
collected.extend(pd_samples)
print(f"  + Ingested {len(pd_samples)} field samples from PlantDoc")

# Mendeley Datasets (Field)
if mendeley_paths.get('potato'):
    pot_samples = discover_images(mendeley_paths['potato'], MENDELEY_POTATO_MAP, 'field', 'MendeleyPotato', global_hashes)
    collected.extend(pot_samples)
    print(f"  + Ingested {len(pot_samples)} field samples from Mendeley Potato")
if mendeley_paths.get('corn'):
    crn_samples = discover_images(mendeley_paths['corn'], MENDELEY_CORN_MAP, 'field', 'MendeleyCorn', global_hashes)
    collected.extend(crn_samples)
    print(f"  + Ingested {len(crn_samples)} field samples from Mendeley Corn")
if mendeley_paths.get('tomato'):
    tom_samples = discover_images(mendeley_paths['tomato'], MENDELEY_TOMATO_MAP, 'field', 'MendeleyTomato', global_hashes)
    collected.extend(tom_samples)
    print(f"  + Ingested {len(tom_samples)} field samples from Mendeley Tomato")

# 5. Anti-Leakage Partition & 50/50 Field-Lab Balancing
rng = random.Random(42)
class_pools = {c: {'lab': [], 'field': []} for c in CANONICAL_CLASSES}
for s in collected:
    class_pools[s['canonical_class']][s['domain']].append(s)

tr_manifest, va_manifest, te_manifest = [], [], []
summary_stats = {}

max_train_per_class = 700
target_total = int(round(max_train_per_class / 0.70))
target_lab = target_total // 2
target_field = target_total - target_lab

for c in CANONICAL_CLASSES:
    lab_pool = sorted(class_pools[c]['lab'], key=lambda x: x['sha256'])
    field_pool = sorted(class_pools[c]['field'], key=lambda x: x['sha256'])
    rng.shuffle(lab_pool)
    rng.shuffle(field_pool)
    
    sel_lab = lab_pool[:target_lab]
    sel_field = field_pool[:target_field]
    # Domain backfilling if needed
    if len(sel_lab) < target_lab and len(field_pool) > target_field:
        sel_field.extend(field_pool[target_field : target_field + (target_lab - len(sel_lab))])
    elif len(sel_field) < target_field and len(lab_pool) > target_lab:
        sel_lab.extend(lab_pool[target_lab : target_lab + (target_field - len(sel_field))])
        
    def split_p(p):
        tr = int(round(len(p) * 0.70))
        va = int(round(len(p) * 0.15))
        return p[:tr], p[tr : tr + va], p[tr + va :]
        
    tr_l, va_l, te_l = split_p(sel_lab)
    tr_f, va_f, te_f = split_p(sel_field)
    
    tr_manifest.extend(tr_l + tr_f)
    va_manifest.extend(va_l + va_f)
    te_manifest.extend(te_l + te_f)
    
    summary_stats[c] = {
        'train_lab': len(tr_l),
        'train_field': len(tr_f),
        'val': len(va_l) + len(va_f),
        'test': len(te_l) + len(te_f),
        'total': len(tr_l) + len(tr_f) + len(va_l) + len(va_f) + len(te_l) + len(te_f)
    }

# Verify Zero Leakage
tr_h = {s['sha256'] for s in tr_manifest}
va_h = {s['sha256'] for s in va_manifest}
te_h = {s['sha256'] for s in te_manifest}
assert not tr_h.intersection(va_h), 'Leakage Train-Val!'
assert not tr_h.intersection(te_h), 'Leakage Train-Test!'
assert not va_h.intersection(te_h), 'Leakage Val-Test!'

# Save manifests
mf_dir = Path("/content/manifests")
mf_dir.mkdir(parents=True, exist_ok=True)
with open(mf_dir / 'train_manifest.json', 'w') as f: json.dump(tr_manifest, f, indent=2)
with open(mf_dir / 'val_manifest.json', 'w') as f: json.dump(va_manifest, f, indent=2)
with open(mf_dir / 'test_manifest.json', 'w') as f: json.dump(te_manifest, f, indent=2)
with open(mf_dir / 'summary.json', 'w') as f: json.dump(summary_stats, f, indent=2)

print("\n" + "=" * 75)
print(f"🎉 INGESTION COMPLETE: {len(tr_manifest)} Train | {len(va_manifest)} Val | {len(te_manifest)} Test")
print(f"Zero-Leakage Hash Guarantee: PASS (0 overlapping images)")
print("=" * 75)
for c, st in summary_stats.items():
    print(f"{c:<36} | Train Lab: {st['train_lab']:>3} | Train Field: {st['train_field']:>3} | Total: {st['total']:>4}")
print("=" * 75)

MANIFEST_DIR = mf_dir
DATASET_ROOT = None  # Paths in manifest are already absolute Linux paths

## 3. Architecture Definition: Dual-Head MobileNetV3-Large

In [ ]:
from typing import Tuple, Dict, Any, List, Optional
import torch.nn as nn
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from sklearn.metrics import f1_score, accuracy_score

class MultiTaskPlantClassifier(nn.Module):
    def __init__(self, num_crops: int = 4, num_diseases: int = 8, pretrained: bool = True, dropout_rate: float = 0.2):
        super().__init__()
        weights = MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
        base_model = mobilenet_v3_large(weights=weights)
        self.features = base_model.features
        self.avgpool = base_model.avgpool
        in_features = getattr(base_model.classifier[0], "in_features", 960)

        # Head 1: Crop Species (Macro-morphology: 4 classes)
        self.crop_head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.Hardswish(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_crops),
        )
        # Head 2: Disease Pathology (Micro-lesions: 8 classes)
        self.disease_head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.Hardswish(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, num_diseases),
        )

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.features(x)
        feat = self.avgpool(feat)
        return torch.flatten(feat, 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feat = self.extract_features(x)
        return self.crop_head(feat), self.disease_head(feat)

class MultiTaskLoss(nn.Module):
    def __init__(self, weight_crop: float = 0.4, weight_disease: float = 0.6, label_smoothing: float = 0.1):
        super().__init__()
        self.weight_crop = weight_crop
        self.weight_disease = weight_disease
        self.criterion_crop = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.criterion_disease = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self, crop_logits, disease_logits, crop_targets, disease_targets):
        l_crop = self.criterion_crop(crop_logits, crop_targets)
        l_dis = self.criterion_disease(disease_logits, disease_targets)
        total = (self.weight_crop * l_crop) + (self.weight_disease * l_dis)
        return total, {"loss_crop": l_crop.detach(), "loss_disease": l_dis.detach(), "total_loss": total.detach()}

## 4. Dataset Loading & Cross-Platform In-Field Augmentations

In [ ]:
from pathlib import Path
from typing import Optional, Union
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.20), ratio=(0.3, 3.3), value="random"),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ColabMultiTaskDataset(Dataset):
    def __init__(self, manifest_path: Union[str, Path], transform=None):
        with open(manifest_path, 'r', encoding='utf-8') as f:
            self.records = json.load(f)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        p = Path(rec["file_path"])
        with Image.open(p) as img:
            img = img.convert("RGB")
            x = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return x, torch.tensor(rec["crop_idx"], dtype=torch.long), torch.tensor(rec["disease_idx"], dtype=torch.long)

## 5. Training Loop with Linear Warmup, AdamW, & Cosine Annealing

In [ ]:
TRAIN_MANIFEST = MANIFEST_DIR / "train_manifest.json"
VAL_MANIFEST = MANIFEST_DIR / "val_manifest.json"

train_ds = ColabMultiTaskDataset(TRAIN_MANIFEST, transform=train_transform)
val_ds = ColabMultiTaskDataset(VAL_MANIFEST, transform=val_transform)

use_pin_memory = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=use_pin_memory)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=use_pin_memory)

model = MultiTaskPlantClassifier(num_crops=4, num_diseases=8, pretrained=True).to(device)
criterion = MultiTaskLoss(weight_crop=0.4, weight_disease=0.6, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

epochs = 25
warmup_epochs = 3
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs - warmup_epochs, 1), eta_min=1e-6)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])

best_joint_acc = 0.0
history = []

print(f"🚀 Starting training on {len(train_ds)} samples across 12 classes ({epochs} epochs)...\n")
for epoch in range(1, epochs + 1):
    # Train
    model.train()
    train_loss = 0.0
    tr_cp, tr_ct, tr_dp, tr_dt = [], [], [], []
    for imgs, crops, dis in train_loader:
        imgs, crops, dis = imgs.to(device), crops.to(device), dis.to(device)
        optimizer.zero_grad()
        c_log, d_log = model(imgs)
        loss, _ = criterion(c_log, d_log, crops, dis)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
        tr_cp.extend(torch.argmax(c_log, dim=-1).cpu().tolist())
        tr_ct.extend(crops.cpu().tolist())
        tr_dp.extend(torch.argmax(d_log, dim=-1).cpu().tolist())
        tr_dt.extend(dis.cpu().tolist())
    scheduler.step()

    tr_joint_acc = sum(1 for cp, ct, dp, dt in zip(tr_cp, tr_ct, tr_dp, tr_dt) if cp == ct and dp == dt) / len(tr_ct)

    # Validation
    model.eval()
    val_loss = 0.0
    val_cp, val_ct, val_dp, val_dt = [], [], [], []
    with torch.no_grad():
        for imgs, crops, dis in val_loader:
            imgs, crops, dis = imgs.to(device), crops.to(device), dis.to(device)
            c_log, d_log = model(imgs)
            loss, _ = criterion(c_log, d_log, crops, dis)
            val_loss += loss.item() * imgs.size(0)
            val_cp.extend(torch.argmax(c_log, dim=-1).cpu().tolist())
            val_ct.extend(crops.cpu().tolist())
            val_dp.extend(torch.argmax(d_log, dim=-1).cpu().tolist())
            val_dt.extend(dis.cpu().tolist())

    crop_acc = accuracy_score(val_ct, val_cp)
    disease_acc = accuracy_score(val_dt, val_dp)
    joint_acc = sum(1 for cp, ct, dp, dt in zip(val_cp, val_ct, val_dp, val_dt) if cp == ct and dp == dt) / len(val_ct)
    dis_f1 = f1_score(val_dt, val_dp, average="macro", zero_division=0)

    epoch_stats = {
        "epoch": epoch,
        "train_loss": train_loss / len(train_ds),
        "train_joint_acc": tr_joint_acc,
        "val_loss": val_loss / len(val_ds),
        "val_crop_acc": crop_acc,
        "val_disease_acc": disease_acc,
        "val_joint_acc": joint_acc,
        "val_disease_f1": dis_f1,
    }
    history.append(epoch_stats)

    print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {epoch_stats['train_loss']:.4f} (Joint: {tr_joint_acc*100:.1f}%) | Val Loss: {epoch_stats['val_loss']:.4f} | Val Joint Acc: {joint_acc*100:.2f}% | Dis F1: {dis_f1*100:.2f}%")
    if joint_acc > best_joint_acc:
        best_joint_acc = joint_acc
        ckpt = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_score": joint_acc,
            "metrics": epoch_stats,
        }
        torch.save(ckpt, "best_multitask_model.pth")
        print(f"  --> [SAVED] New best model checkpoint: Joint Acc = {joint_acc*100:.2f}%")

## 6. Training Curve Visualization

In [ ]:
import matplotlib.pyplot as plt

ep_list = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(ep_list, [h["train_loss"] for h in history], label="Train Loss", color="tab:blue")
axes[0].plot(ep_list, [h["val_loss"] for h in history], label="Val Loss", color="tab:red", linestyle="--")
axes[0].set_title("Multi-Task Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(ep_list, [h["train_joint_acc"]*100 for h in history], label="Train Joint Acc (%)", color="tab:blue")
axes[1].plot(ep_list, [h["val_joint_acc"]*100 for h in history], label="Val Joint Acc (%)", color="tab:green")
axes[1].plot(ep_list, [h["val_disease_f1"]*100 for h in history], label="Val Disease Macro F1 (%)", color="tab:orange", linestyle=":")
axes[1].set_title("Multi-Task Joint Accuracy & Macro F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Percentage (%)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. Reload Best Checkpoint & Export to ONNX (with Parity Verification)

In [ ]:
# Reload the best model checkpoint
best_ckpt = torch.load("best_multitask_model.pth", map_location="cpu")
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()
model.cpu()
print(f"Loaded best model from Epoch {best_ckpt['epoch']} with Joint Accuracy: {best_ckpt['best_score']*100:.2f}%")

onnx_path = "agritech_multitask.onnx"
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["crop_logits", "disease_logits"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "crop_logits": {0: "batch_size"},
        "disease_logits": {0: "batch_size"},
    },
    dynamo=False,
)
print(f"Successfully exported ONNX model to {onnx_path}")

# Verify with ONNX Runtime
import onnxruntime as ort
session = ort.InferenceSession(onnx_path)
ort_inputs = {"input": dummy_input.numpy()}
ort_crops, ort_dis = session.run(None, ort_inputs)

with torch.no_grad():
    pt_crops, pt_dis = model(dummy_input)

crop_diff = np.max(np.abs(ort_crops - pt_crops.numpy()))
dis_diff = np.max(np.abs(ort_dis - pt_dis.numpy()))
print(f"ONNX Runtime numerical parity verified! Max Crop diff: {crop_diff:.6f}, Max Disease diff: {dis_diff:.6f}")

## 8. Test Set Evaluation & In-Field Domain Disaggregation Reporting

In [ ]:
TEST_MANIFEST = MANIFEST_DIR / "test_manifest.json"

with open(TEST_MANIFEST, 'r') as f:
    test_records = json.load(f)

test_ds = ColabMultiTaskDataset(TEST_MANIFEST, transform=val_transform)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

model.eval()
model.to(device)

all_cp, all_ct, all_dp, all_dt = [], [], [], []
with torch.no_grad():
    for imgs, crops, dis in test_loader:
        imgs = imgs.to(device)
        c_log, d_log = model(imgs)
        all_cp.extend(torch.argmax(c_log, dim=-1).cpu().tolist())
        all_ct.extend(crops.tolist())
        all_dp.extend(torch.argmax(d_log, dim=-1).cpu().tolist())
        all_dt.extend(dis.tolist())

test_crop_acc = accuracy_score(all_ct, all_cp)
test_dis_acc = accuracy_score(all_dt, all_dp)
test_joint_acc = sum(1 for cp, ct, dp, dt in zip(all_cp, all_ct, all_dp, all_dt) if cp == ct and dp == dt) / len(all_ct)
test_dis_f1 = f1_score(all_dt, all_dp, average="macro", zero_division=0)

# Domain Disaggregation: Field vs Lab
field_indices = [i for i, r in enumerate(test_records) if r.get('domain') == 'field']
lab_indices   = [i for i, r in enumerate(test_records) if r.get('domain') == 'lab']

def compute_subset_acc(indices):
    if not indices: return 0.0, 0.0, 0.0
    sub_ct = [all_ct[i] for i in indices]
    sub_cp = [all_cp[i] for i in indices]
    sub_dt = [all_dt[i] for i in indices]
    sub_dp = [all_dp[i] for i in indices]
    c_acc = accuracy_score(sub_ct, sub_cp)
    d_acc = accuracy_score(sub_dt, sub_dp)
    j_acc = sum(1 for cp, ct, dp, dt in zip(sub_cp, sub_ct, sub_dp, sub_dt) if cp == ct and dp == dt) / len(indices)
    return c_acc, d_acc, j_acc

f_cacc, f_dacc, f_jacc = compute_subset_acc(field_indices)
l_cacc, l_dacc, l_jacc = compute_subset_acc(lab_indices)

print("=" * 75)
print("🌟 FINAL HELD-OUT TEST BENCHMARK RESULTS (AUTHENTIC MULTI-DATASET)")
print("=" * 75)
print(f"Total Test Samples Evaluated: {len(all_ct)}")
print(f"Overall Crop Accuracy:        {test_crop_acc*100:.2f}%")
print(f"Overall Disease Accuracy:     {test_dis_acc*100:.2f}%")
print(f"Overall Joint Exact Match:    {test_joint_acc*100:.2f}%")
print(f"Overall Disease Macro F1:     {test_dis_f1*100:.2f}%")
print("-" * 75)
print("DOMAIN DISAGGREGATION BREAKDOWN:")
print(f"  Authentic Field Samples ({len(field_indices)}): Joint Acc = {f_jacc*100:.2f}% | Disease Acc = {f_dacc*100:.2f}%")
print(f"  Clean Lab Samples       ({len(lab_indices)}): Joint Acc = {l_jacc*100:.2f}% | Disease Acc = {l_dacc*100:.2f}%")
print("=" * 75)

## 9. Download Trained Model Weights and ONNX Artifacts

In [ ]:
try:
    from google.colab import files
    print("\n[DOWNLOAD] Triggering browser download for trained artifacts...")
    files.download("agritech_multitask.onnx")
    files.download("best_multitask_model.pth")
    files.download("/content/manifests/summary.json")
    print("Downloads initiated. If your browser blocks multiple popups, right-click the files on the left to download!")
except Exception as e:
    print(f"\n[INFO] Auto-download note: {e}")
    print("Files are saved at /content/agritech_multitask.onnx and /content/best_multitask_model.pth")